[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/safe-t-ai/safe-t-ai.github.io/blob/main/notebooks/05_test4_suppressed_demand.ipynb)

# Test 4: Suppressed Demand Analysis

Runs `SuppressedDemandAnalyzer` to model potential vs. actual active-transportation demand
across Durham census tracts, exports audit artifacts, and generates all frontend static files
including `data-manifest.json` and `metadata.json` (final step in the pipeline).

## 1. Install dependencies

In [1]:
!pip install -q geopandas

## 2. Bootstrap repo

In [2]:
import subprocess
import sys
from pathlib import Path

NOTEBOOKS_DIR = Path("/content/safe-t-ai.github.io/notebooks")
if not NOTEBOOKS_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth=1",
         "https://github.com/safe-t-ai/safe-t-ai.github.io.git",
         "/content/safe-t-ai.github.io"],
        check=True,
    )

if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))

from colab_utils import prepare_notebook, publish_artifacts, save_notebook

REPO = prepare_notebook(pull_latest=True)
print(f"Repo root: {REPO}")

Repo root: /content/safe-t-ai.github.io


## 3. Load census and infrastructure data

In [3]:
import json
import geopandas as gpd

from config import (
    HIGH_SUPPRESSION_THRESHOLD, RAW_DATA_DIR, SIMULATED_DATA_DIR,
    SUPPRESSED_DEMAND_CONFIG, CENSUS_VINTAGE, CRASH_ANALYSIS_YEARS,
    PLAUSIBILITY_RANGES,
)
from models.demand_analyzer import SuppressedDemandAnalyzer
from utils.data_loading import load_infrastructure_data

census_path = RAW_DATA_DIR / "durham_census_tracts.geojson"
if not census_path.exists():
    raise FileNotFoundError(f"Census data not found at {census_path}. Run fetch_durham_data.py first.")

census_gdf = gpd.read_file(census_path)
print(f"Loaded {len(census_gdf)} census tracts")

infrastructure_df = load_infrastructure_data()
print(f"Loaded infrastructure scores for {len(infrastructure_df)} tracts")

Loaded 67 census tracts
Loaded infrastructure scores for 67 tracts


## 4. Run suppressed demand analysis

In [4]:
analyzer = SuppressedDemandAnalyzer(census_gdf, infrastructure_df)
results = analyzer.run_analysis()

summary = results["summary"]
print(f"Total potential demand: {summary['total_potential_demand']:,} trips/day")
print(f"Total actual demand:    {summary['total_actual_demand']:,} trips/day")
print(f"Total suppressed:       {summary['total_suppressed_demand']:,} trips/day")
print(f"Overall suppression:    {summary['suppression_rate']:.1f}%")
print(f"High-suppression tracts (>{HIGH_SUPPRESSION_THRESHOLD}%): {summary['high_suppression_tracts']}")

print("\nDemand by income quintile:")
print(f"{'Quintile':<15} {'Potential':>10} {'Actual':>10} {'Suppressed':>10} {'Supp %':>8}")
print("-" * 60)
for quintile, metrics in results["by_quintile"].items():
    print(
        f"{quintile:<15} {metrics['potential_demand']:>10.1f} "
        f"{metrics['actual_demand']:>10.1f} "
        f"{metrics['suppressed_demand']:>10.1f} "
        f"{metrics['suppression_pct']:>7.1f}%"
    )

Total potential demand: 44,314 trips/day
Total actual demand:    8,463 trips/day
Total suppressed:       35,850 trips/day
Overall suppression:    80.9%
High-suppression tracts (>70%): 50

Demand by income quintile:
Quintile         Potential     Actual Suppressed   Supp %
------------------------------------------------------------
Q1 (Poorest)         508.5       83.7      424.8    82.7%
Q2                   684.2      117.6      566.6    79.9%
Q3                   701.4      138.7      562.7    79.8%
Q4                   750.3      154.3      595.9    77.6%
Q5 (Richest)         673.6      139.6      533.9    73.5%


## 5. AI detection scorecard

In [5]:
naive = results["detection_scorecard"]["naive_ai"]
soph = results["detection_scorecard"]["sophisticated_ai"]
expert = results["detection_scorecard"]["human_expert_baseline"]

print(f"{'Model':<20} {'Correlation':>12} {'RMSE':>10} {'Q1 Bias':>10} {'Detection %':>12}")
print("-" * 70)
print(f"{'Naive AI':<20} {naive['correlation_with_potential']:>12.3f} {naive['rmse']:>10.1f} {naive['bias_q1']:>9.1f}% {naive['detection_rate_high_suppression']:>11.1f}%")
print(f"{'Sophisticated AI':<20} {soph['correlation_with_potential']:>12.3f} {soph['rmse']:>10.1f} {soph['bias_q1']:>9.1f}% {soph['detection_rate_high_suppression']:>11.1f}%")
print(f"{'Human Expert':<20} {expert['correlation_with_potential']:>12.3f} {expert['rmse']:>10.1f} {expert['bias_q1']:>9.1f}% {expert['detection_rate_high_suppression']:>11.1f}%")

Model                 Correlation       RMSE    Q1 Bias  Detection %
----------------------------------------------------------------------
Naive AI                    0.248      603.1     -83.5%         0.0%
Sophisticated AI            0.820      318.4     -43.9%       100.0%
Human Expert                0.850       60.0      -5.0%        80.0%


## 6. Export backend simulated files

In [6]:
SIMULATED_DATA_DIR.mkdir(parents=True, exist_ok=True)

q1_suppression = results["by_quintile"]["Q1 (Poorest)"]["suppression_pct"]
q5_suppression = results["by_quintile"]["Q5 (Richest)"]["suppression_pct"]

demand_report = {
    "_provenance": {
        "data_type": "mixed",
        "real": ["infrastructure scores (OpenStreetMap)"],
        "simulated": ["potential demand", "actual demand", "AI detection models"],
        "parameters": {
            "high_suppression_threshold": HIGH_SUPPRESSION_THRESHOLD,
            "income_factor_slope": SUPPRESSED_DEMAND_CONFIG["income_factor_slope"],
            "destination_factor_min": SUPPRESSED_DEMAND_CONFIG["destination_factor_min"],
            "destination_factor_range": SUPPRESSED_DEMAND_CONFIG["destination_factor_range"],
            "population_proxy_coeff": SUPPRESSED_DEMAND_CONFIG["population_proxy_coeff"],
            "infrastructure_adjustment_coeff": SUPPRESSED_DEMAND_CONFIG["infrastructure_adjustment_coeff"],
            "detection_threshold_multiplier": SUPPRESSED_DEMAND_CONFIG["detection_threshold_multiplier"],
        },
    },
    "summary": results["summary"],
    "high_suppression_threshold": HIGH_SUPPRESSION_THRESHOLD,
    "by_quintile": results["by_quintile"],
    "findings": [
        f"{summary['suppression_rate']:.1f}% of potential active transportation demand is suppressed by poor infrastructure",
        f"Q1 (poorest) areas have {q1_suppression:.1f}% suppression vs {q5_suppression:.1f}% in Q5",
        f"Naive AI has only {naive['correlation_with_potential']:.2f} correlation with potential demand (fails to detect suppressed demand)",
        f"Sophisticated AI achieves {soph['correlation_with_potential']:.2f} correlation but still undercounts by {abs(soph['bias_q1']):.1f}% in Q1",
        "AI tools measuring only observed demand miss suppressed demand—low-income areas where poor infrastructure collapses actual trips toward zero",
    ],
}

with open(SIMULATED_DATA_DIR / "demand_analysis.json", "w") as f:
    json.dump(demand_report, f, indent=2)
print("Exported demand_analysis.json")

with open(SIMULATED_DATA_DIR / "demand_funnel.json", "w") as f:
    json.dump(results["funnel_data"], f, indent=2)
print("Exported demand_funnel.json")

with open(SIMULATED_DATA_DIR / "correlation_matrix.json", "w") as f:
    json.dump(results["correlation_matrix"], f, indent=2)
print("Exported correlation_matrix.json")

with open(SIMULATED_DATA_DIR / "detection_scorecard.json", "w") as f:
    json.dump(results["detection_scorecard"], f, indent=2)
print("Exported detection_scorecard.json")

with open(SIMULATED_DATA_DIR / "network_flow.json", "w") as f:
    json.dump(results["network_flow"], f, indent=2)
print("Exported network_flow.json")

Exported demand_analysis.json
Exported demand_funnel.json
Exported correlation_matrix.json
Exported detection_scorecard.json
Exported network_flow.json


## 7. Export demand_geo_data.json

In [7]:
demand_data = results["demand_data"]

demand_geo = census_gdf[["tract_id", "geometry"]].merge(
    demand_data[[
        "tract_id", "potential_demand", "actual_demand", "suppressed_demand",
        "suppression_pct", "infrastructure_score", "income_quintile"
    ]],
    on="tract_id"
)
demand_geo["geometry"] = demand_geo["geometry"].simplify(0.001)

demand_geo_dict = json.loads(demand_geo.to_json())
with open(SIMULATED_DATA_DIR / "demand_geo_data.json", "w") as f:
    json.dump(demand_geo_dict, f)
print(f"Exported demand_geo_data.json ({len(demand_geo)} tracts)")

Exported demand_geo_data.json (67 tracts)


## 8. Copy demand files to frontend

In [8]:
frontend_data_dir = REPO / "frontend" / "public" / "data"
frontend_data_dir.mkdir(parents=True, exist_ok=True)

def copy_json(src, dst, indent=2):
    with open(src) as f:
        data = json.load(f)
    with open(dst, "w") as f:
        json.dump(data, f, indent=indent)
    return data

copy_json(SIMULATED_DATA_DIR / "demand_analysis.json", frontend_data_dir / "demand-report.json")
copy_json(SIMULATED_DATA_DIR / "demand_funnel.json", frontend_data_dir / "demand-funnel.json")
copy_json(SIMULATED_DATA_DIR / "detection_scorecard.json", frontend_data_dir / "detection-scorecard.json")
copy_json(SIMULATED_DATA_DIR / "demand_geo_data.json", frontend_data_dir / "demand-geo-data.json", indent=None)
print("Frontend demand files written.")

Frontend demand files written.


## 9. Generate data-manifest.json and metadata.json

In [9]:
import os
from datetime import datetime, timezone
from utils.freshness import read_meta

census_meta = read_meta(RAW_DATA_DIR / "durham_census_tracts.geojson")
crash_meta = read_meta(RAW_DATA_DIR / "ncdot_nonmotorist_durham.csv")
osm_meta = read_meta(RAW_DATA_DIR / "osm_infrastructure.json")

analysis_range = f"{min(CRASH_ANALYSIS_YEARS)}-{max(CRASH_ANALYSIS_YEARS)}"
census_coverage = f"{CENSUS_VINTAGE - 4}-{CENSUS_VINTAGE}"

manifest = {
    "sources": {
        "census_demographics": {
            "type": "real",
            "provider": f"US Census Bureau ACS {CENSUS_VINTAGE}",
            "temporal_coverage": census_coverage,
            "fetched_at": (census_meta or {}).get("fetched_at"),
            "files": ["census-tracts.json", "choropleth-data.json"],
        },
        "crash_volumes": {
            "type": "real",
            "provider": "NCDOT Non-Motorist Crash Database (ArcGIS)",
            "temporal_coverage": analysis_range,
            "fetched_at": (crash_meta or {}).get("fetched_at"),
            "files": [
                "crash-report.json", "crash-time-series.json",
                "confusion-matrices.json", "crash-geo-data.json",
            ],
        },
        "volume_predictions": {
            "type": "simulated",
            "rationale": "Strava Metro / StreetLight Data require institutional license",
            "files": [
                "volume-report.json", "counter-locations.json",
                "accuracy-by-income.json", "accuracy-by-race.json",
                "scatter-data.json",
            ],
        },
        "infrastructure_recommendations": {
            "type": "mixed",
            "real_source": "OpenStreetMap infrastructure density per tract",
            "fetched_at": (osm_meta or {}).get("fetched_at"),
            "rationale": "Project selection uses real OSM infrastructure gaps; danger scores and allocation logic are simulated",
            "files": [
                "infrastructure-report.json", "danger-scores.json",
                "budget-allocation.json", "recommendations.json",
            ],
        },
        "suppressed_demand": {
            "type": "mixed",
            "real_source": "OpenStreetMap infrastructure density per tract",
            "fetched_at": (osm_meta or {}).get("fetched_at"),
            "rationale": "Infrastructure scores from OSM; demand modeling and AI detection are simulated",
            "files": [
                "demand-report.json", "demand-funnel.json",
                "detection-scorecard.json", "demand-geo-data.json",
            ],
        },
    },
    "plausibility_ranges": PLAUSIBILITY_RANGES,
}

with open(frontend_data_dir / "data-manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)
print("Wrote data-manifest.json")

metadata = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "data_hash": os.environ.get("DATA_HASH", "local"),
    "github_run_url": os.environ.get("GITHUB_RUN_URL", ""),
    "git_sha": os.environ.get("GIT_SHA", ""),
    "sources": {
        "real": [
            f"US Census ACS {CENSUS_VINTAGE}",
            "OpenStreetMap infrastructure inventory",
            "NCDOT non-motorist crashes (ArcGIS)",
        ],
        "simulated": ["volume predictions", "infrastructure recommendations", "demand analysis"],
    },
}

with open(frontend_data_dir / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)
print("Wrote metadata.json")

Wrote data-manifest.json
Wrote metadata.json


## 10. Publish artifacts

In [ ]:
notebook_path = save_notebook("05_test4_suppressed_demand.ipynb", repo_dir=REPO)

paths = [
    "backend/data/simulated/demand_analysis.json",
    "backend/data/simulated/demand_funnel.json",
    "backend/data/simulated/correlation_matrix.json",
    "backend/data/simulated/detection_scorecard.json",
    "backend/data/simulated/network_flow.json",
    "backend/data/simulated/demand_geo_data.json",
    "frontend/public/data/demand-report.json",
    "frontend/public/data/demand-funnel.json",
    "frontend/public/data/detection-scorecard.json",
    "frontend/public/data/demand-geo-data.json",
    "frontend/public/data/data-manifest.json",
    "frontend/public/data/metadata.json",
]
if notebook_path:
    paths.insert(0, notebook_path)

publish_artifacts(
    paths,
    message="data: regenerate Test 4 suppressed demand + manifest",
    repo_dir=REPO,
)